In [ ]:
# 1. Install Ollama and Pinggy
!apt-get update && apt-get install zstd -y
!curl -fsSL https://ollama.com/install.sh | sh
!pip install pinggy

import os
import subprocess
import time

# 2. Start Ollama with the correct settings
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS'] = '*'

print("--> Starting Ollama...")
with open("ollama.log", "w") as log_file:
    subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file)


In [ ]:
import os
import subprocess
import time
import requests
import pinggy

# 1. Setup Environment
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS'] = '*'

# 2. Restart Ollama to clear previous "Address already in use" errors
print("--> Cleaning up existing processes...")
!pkill ollama
time.sleep(2)

print("--> Starting Ollama...")
with open("ollama.log", "w") as log_file:
    subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file)

# 3. Wait for Ollama to be ready
print("--> Waiting for Ollama to wake up...")
for i in range(20):
    try:
        if requests.get("http://localhost:11434/").status_code == 200:
            print("--> Ollama is ready!")
            break
    except:
        time.sleep(1)

# 4. Pull the model (this can take a minute)
print("--> Pulling Gemma (please wait)...")
!ollama pull gemma4:e2b

# 5. Start Tunnel with a "Wait for URL" loop
print("--> Initializing Tunnel...")
tunnel = pinggy.start_tunnel(
    forwardto="localhost:11434",
    # Corrected header modification format for the Python SDK
    headermodification=[{"type": "update", "key": "Host", "value": "localhost:11434"}]
)

# 6. CRITICAL: Wait for the URL to actually generate
print("--> Fetching your public URL...")
public_url = None
for _ in range(15): # Try for 15 seconds
    if tunnel.urls:
        public_url = tunnel.urls[0]
        break
    time.sleep(1)

if public_url:
    print("\n" + "="*50)
    print("CONNECTED SUCCESSFULLY!")
    print(f"Your Public URL: {public_url}")
    print("="*50 + "\n")
else:
    print("--> Error: Tunnel failed to provide a URL. Check 'ollama.log' for errors.")

!nvidia-smi